# DSPy

**Module:** 09-llm-frameworks

**Notebook:** `05-dspy.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **What is DSPy?** with clear contracts and failure modes
- Explain and apply **Signatures** with clear contracts and failure modes
- Explain and apply **Modules** with clear contracts and failure modes
- Explain and apply **Optimizers** with clear contracts and failure modes
- Explain and apply **Why Teams Adopt DSPy** with clear contracts and failure modes
- Explain and apply **Minimal Program Sketch** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — DSPy

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **What is DSPy?**
2. **Signatures**
3. **Modules**
4. **Optimizers**
5. **Why Teams Adopt DSPy**
6. **Minimal Program Sketch**

Read top-to-bottom once, then revisit weak spots with the exercises.


## What is DSPy?

### Definition
**What is DSPy?** is a core building block in 05-dspy within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around What is DSPy? typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For What is DSPy?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain What is DSPy? as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating What is DSPy? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for What is DSPy?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use What is DSPy? when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does What is DSPy? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "What is DSPy?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "What is DSPy?"
    notebook: str = "05-dspy"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


## Signatures

### Definition
**Signatures** is a core building block in 05-dspy within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Signatures typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Signatures: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Signatures as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Signatures as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Signatures
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Signatures when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Signatures" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Signatures"
    notebook: str = "05-dspy"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Signatures"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Signatures"}
strong = {"definition": "Signatures", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Signatures"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Signatures", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Signatures

**Situation:** A team wants to productionize a feature involving **Signatures**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Modules

### Definition
**Modules** is a core building block in 05-dspy within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Modules typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Modules: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Modules as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Modules as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Modules
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Modules when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Modules" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Modules"
    notebook: str = "05-dspy"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Modules"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Modules"}
strong = {"definition": "Modules", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Modules"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Modules", "passed": len(checks)-len(failed), "failed": failed})


## Optimizers

### Definition
**Optimizers** is a core building block in 05-dspy within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Optimizers typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Optimizers: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Optimizers as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Optimizers as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Optimizers
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Optimizers when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Optimizers" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Optimizers"
    notebook: str = "05-dspy"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
GOLDEN = [{"x": "charged twice", "y": "billing"}, {"x": "SSO", "y": "auth"}]

def predict(x: str) -> str:
    return "billing" if "charge" in x.lower() or "invoice" in x.lower() else ("auth" if "sso" in x.lower() or "login" in x.lower() else "other")

def eval_prompt(name: str):
    rows = [(g["x"], g["y"], predict(g["x"])) for g in GOLDEN]
    acc = sum(y == p for _, y, p in rows) / len(rows)
    return {"name": name, "accuracy": acc, "rows": rows}

print(eval_prompt("v1"))


In [ ]:
import hashlib

def prompt_version(text: str) -> str:
    return "pv_" + hashlib.sha1(text.encode()).hexdigest()[:10]

a = "Label tickets carefully"
b = "Label tickets as billing|auth|outage|other. JSON only."
print(prompt_version(a), prompt_version(b))


### Worked scenario — Optimizers

**Situation:** A team wants to productionize a feature involving **Optimizers**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Why Teams Adopt DSPy

### Definition
**Why Teams Adopt DSPy** is a core building block in 05-dspy within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Why Teams Adopt DSPy typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Why Teams Adopt DSPy: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Why Teams Adopt DSPy as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Why Teams Adopt DSPy as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Why Teams Adopt DSPy
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Why Teams Adopt DSPy when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Why Teams Adopt DSPy" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Why Teams Adopt DSPy"
    notebook: str = "05-dspy"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Framework mental model: compose runnable steps
class Runnable:
    def __init__(self, fn): self.fn = fn
    def __or__(self, other):
        return Runnable(lambda x: other.fn(self.fn(x)))
    def invoke(self, x): return self.fn(x)

prompt = Runnable(lambda q: f"Answer briefly: {q}")
model = Runnable(lambda p: f"<model-output for: {p[:40]}...>")
parse = Runnable(lambda t: {"text": t, "n": len(t)})
chain = prompt | model | parse
print(chain.invoke("What is LCEL?"))


In [ ]:
# Index → retrieve → generate sketch (RAG-shaped)
NODES = [{"id": 1, "text": "LCEL composes runnables with |"}, {"id": 2, "text": "LlamaIndex focuses on data indexes"}]

def retrieve(q, k=1):
    return sorted(NODES, key=lambda n: len(set(q.lower().split()) & set(n["text"].lower().split())), reverse=True)[:k]

def generate(q, nodes):
    ctx = " | ".join(n["text"] for n in nodes)
    return f"Q: {q}\nCTX: {ctx}\nA: Based on context, {nodes[0]['text']}."

print(generate("What is LCEL?", retrieve("LCEL compose")))


## Minimal Program Sketch

### Definition
**Minimal Program Sketch** is a core building block in 05-dspy within LLM frameworks. Treat it as an orchestration layer—useful only when it clarifies ownership of steps: something you can name, version, test, and operate.

### Why it matters
In LLM frameworks, weak designs around Minimal Program Sketch typically surface as framework lock-in, opaque magic, and untested compositions. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Minimal Program Sketch: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like runnable chains, indexes, and portable pipelines.

### Intuition
Explain Minimal Program Sketch as an orchestration layer—useful only when it clarifies ownership of steps. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Minimal Program Sketch as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Minimal Program Sketch
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of LLM frameworks: framework lock-in, opaque magic, and untested compositions

### When to use
Use Minimal Program Sketch when your product path depends on this concern in LLM frameworks. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Minimal Program Sketch" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Minimal Program Sketch"
    notebook: str = "05-dspy"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Minimal Program Sketch"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Minimal Program Sketch"}
strong = {"definition": "Minimal Program Sketch", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Minimal Program Sketch"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Minimal Program Sketch", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Minimal Program Sketch

**Situation:** A team wants to productionize a feature involving **Minimal Program Sketch**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **DSPy**.

| Topic | Do | Don't |
|-------|----|-------|
| What is DSPy? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Signatures | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Modules | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Optimizers | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Why Teams Adopt DSPy | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Minimal Program Sketch | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| What is DSPy? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Signatures | Key concept covered in this notebook; see its section for definition and pitfalls |
| Modules | Key concept covered in this notebook; see its section for definition and pitfalls |
| Optimizers | Key concept covered in this notebook; see its section for definition and pitfalls |
| Why Teams Adopt DSPy | Key concept covered in this notebook; see its section for definition and pitfalls |
| Minimal Program Sketch | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **DSPy** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **09-llm-frameworks**.


## Try It Yourself

1. Implement a failing test/fixture for **What is DSPy?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Signatures**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Modules**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Optimizers**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Why Teams Adopt DSPy**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
